# Intro to `gwrefpy`

`gwrefpy` is a Python package for **detecting deviations in groundwater time series**.

This notebook can be downloaded from the source code [here](https://github.com/sgfsweden/gwrefpy/blob/main/docs/user_guide/1_intro_gwrefpy.ipynb).

```{admonition} Problem Statement
:class: note
**The problem:** Groundwater levels rise and fall naturally with weather and the seasons. So when an observation well behaves unexpectedly, how do we tell whether human activity (pumping, construction, drainage) is to blame, the instrument is broken (logger drift, physical interventions), or whether it is just natural variation?

**How `gwrefpy` solves it:** An **observation well** (the one we care about) is compared against one or more **reference wells** that represent undisturbed, natural conditions. `gwrefpy` fits the observation well to the reference wells by regression; once later measurements stop following that fit, a deviation — some kind of influence — has occurred.
```

The two building blocks of `gwrefpy` are **`Model`** — a site or area of interest — and **`Well`** — a location where groundwater data is collected. A `Model` holds any number of `Well` objects, and the two can be created independently and linked together later.

Let's start by importing `gwrefpy` (along with `pandas`, which we use to load the data).

In [ ]:
import pandas as pd

import gwrefpy as gr

# Creating a Model

A **`Model`** represents a site or area of interest and is made up of any number of wells. Here we create one named `"My First gwrefpy Model"`.

In [ ]:
model = gr.Model(name="My First gwrefpy Model")

# Creating Wells

A **`Well`** is a location where groundwater data is collected. Each well is either an *observation* well or a *reference* well.

```{glossary}
*Reference wells*
    Wells that are used to represent natural conditions.

*Observation wells*
    Wells that the user is interested in to see if they are influenced by anthropogenic activities.
```

We will create two observation wells and two reference wells and later add them to our model.

## Creating data timeseries

A well's data is a `pandas` time series. In this example we build those series from CSV files.

```{tip}
Data types can be other than CSV. Check the pandas documentation for inspiration.
```

In [ ]:
base_url = "https://github.com/sgfsweden/gwrefpy/raw/dev/docs/user_guide/data"


def load_series(filename):
    df = pd.read_csv(
        f"{base_url}/{filename}",
        sep=";",
        index_col=0,
        parse_dates=True,
    )
    # Each CSV has a single value column; take it as a pandas Series.
    return df.iloc[:, 0]


obs1_series = load_series("obs1_mod_wel.csv")
ref1_series = load_series("ref1_mod_wel.csv")
obs2_series = load_series("obs2_mod_wel.csv")
ref2_series = load_series("ref2_mod_wel.csv")

Next we create the `Well` objects and attach the time series to them.

```{admonition} The Well object
:class: note

`Well` takes three arguments: a `name` (string), `is_reference` (a boolean choosing reference or observation well), and the `timeseries` (the pandas series we created earlier).
```

In [ ]:
obs1 = gr.Well(name="obs1", is_reference=False, timeseries=obs1_series)
ref1 = gr.Well(name="ref1", is_reference=True, timeseries=ref1_series)
obs2 = gr.Well(name="obs2", is_reference=False, timeseries=obs2_series)
ref2 = gr.Well(name="ref2", is_reference=True, timeseries=ref2_series)

Now we add the wells to the model.

# Adding wells

We add wells with `add_well()`, passing either a single well or a list of wells.

In [ ]:
# Add several wells at once with a list
model.add_well([obs1, ref1, obs2])

# Add a single well
model.add_well(ref2)

Now let's confirm the wells are stored in the model.

# Accessing wells

Wells are located in the `wells` object in our newly created `model`.

The `obs()` and `ref()` methods list the observation and reference wells in the model.

In [ ]:
model.wells.obs()

Check the reference wells:

In [ ]:
model.wells.ref()

We can look up a specific well using its name.

In [ ]:
model.wells["obs1"]

Or multiple wells using filters. For more info, see
{py:meth}`~src.gwrefpy.wellcollection.WellCollection.filter` in the API reference.

In [ ]:
model.wells.filter(name="ref1", kind="ref")

Use `model.wells.to_dataframe()` to get a dataframe view of all model wells.

In [ ]:
model.wells.to_dataframe()

# Removing wells from a model

Remove wells with the `.remove()` method.

In [ ]:
obs1 = model.wells.remove("obs1")  # returns whatever was removed
model.wells.obs()  # now only holds obs2

For large remove operations, it can be valuable first running the operation "dry" and ensure it matches your expectations.

In [ ]:
what_will_be_removed = model.wells.remove(["ref1", "ref2"], dry_run=True)
what_will_be_removed

That's it for the intro to `gwrefpy`.

The next notebooks build on this model to cover the rest of `gwrefpy`: pairing observation and reference wells, fitting them, and plotting the results.